In [0]:
import requests
import json
from uuid import uuid4
from datetime import datetime
import pyspark.sql.functions as F
from pyspark.sql.types import IntegerType

In [0]:
def log(message):
    print(message)

def saveToTable(p_url, p_endpoint, p_headers, p_guid, p_timestamp):
    guid = p_guid if p_guid else str(uuid4())
    ingestion_timestamp = p_timestamp if p_timestamp else datetime.now()
    url = f"{p_url}/{p_endpoint}"
    
    try:
        response = requests.request("GET", url, headers=p_headers)
        response_json = response.json()
        pages = response_json['info']['pages']
        log(f"Saving data to table {p_endpoint}s. Number of pages: {pages}")
    except Exception as e:
        return f"Request failed: {e}"
    
    for page in range(1, pages+1):
        try:
            log(f"Saving page {page}")
            paged_url = f"{url}?page={page}"
            paged_response = requests.request("GET", paged_url, headers=p_headers)
            paged_response_json = paged_response.json()

            df = spark.createDataFrame(paged_response_json['results'])
            df = df.withColumn("guid", F.lit(guid)).withColumn("ingestion_timestamp", F.lit(ingestion_timestamp))
            df.write.mode('append').saveAsTable(f"bronze_{p_endpoint}s")
            log(f"Page {page} saved")
        except Exception as e:
            return f"Request failed: {e}"
    log(f"Data saving to table {p_endpoint}s completed")

def saveToTableFromDelta(source_df, target_table):
    source_df.write.mode('append').saveAsTable(target_table)
    log(f"Data saving to table {target_table} completed")